# Context-Aware Chatbot using Retrieval-Augmented Generation (RAG)

## Problem Statement
Traditional chatbots often generate responses without referencing reliable knowledge sources. This can lead to inaccurate or hallucinated answers.

The goal of this project is to build a **context-aware chatbot** that retrieves relevant information from a custom knowledge base and generates accurate responses using Retrieval-Augmented Generation (RAG).

## Objective
The objective of this project is to develop a chatbot that:

- Retrieves relevant documents from a knowledge base
- Uses embeddings and vector search for information retrieval
- Generates responses using a language model
- Maintains conversational context
- Provides an interactive user interface using Streamlit

Import Required Libraries

In [ ]:
!pip install -U langchain langchain-community sentence-transformers faiss-cpu

In [ ]:
# Import required libraries

import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

## Creating a Knowledge Base

To enable the chatbot to answer questions, we first create a simple knowledge base containing information related to Artificial Intelligence and Machine Learning.

This knowledge base will later be used for retrieving relevant information during user queries.

In [ ]:
import os
# Create knowledge base folder
os.makedirs("knowledge_base", exist_ok=True)

text = """
Artificial Intelligence (AI) is a field of computer science that focuses on building systems capable of performing tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence where systems learn patterns from data and improve their performance over time without being explicitly programmed.

Deep Learning is a specialized area of machine learning that uses neural networks with many layers to learn complex patterns in data.

Natural Language Processing (NLP) is a branch of AI that focuses on enabling computers to understand, interpret, and generate human language.
"""

with open("knowledge_base/ai_ml_notes.txt", "w") as f:
    f.write(text)

print("Knowledge base created successfully.")

Knowledge base created successfully.


## Loading Documents

The next step is to load the documents from the knowledge base.  
These documents will later be processed and converted into vector embeddings.

In [ ]:
loader = TextLoader("knowledge_base/ai_ml_notes.txt")
documents = loader.load()

documents

[Document(metadata={'source': 'knowledge_base/ai_ml_notes.txt'}, page_content='\nArtificial Intelligence (AI) is a field of computer science that focuses on building systems capable of performing tasks that normally require human intelligence.\n\nMachine Learning is a subset of Artificial Intelligence where systems learn patterns from data and improve their performance over time without being explicitly programmed.\n\nDeep Learning is a specialized area of machine learning that uses neural networks with many layers to learn complex patterns in data.\n\nNatural Language Processing (NLP) is a branch of AI that focuses on enabling computers to understand, interpret, and generate human language.\n')]

## Text Chunking

Large documents are difficult for language models to process efficiently.  
Therefore, the text is divided into smaller chunks to improve retrieval performance.

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(len(docs))

4


## Generating Embeddings

Embeddings convert text into numerical vector representations.  
These vectors allow the system to measure semantic similarity between queries and documents.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_3320/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector Database Creation

The embeddings are stored in a FAISS vector database.  
This allows efficient similarity search when the user asks a question.

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)

print("Vector database created successfully.")

Vector database created successfully.


## Testing the Retrieval System

Before building the chatbot, we test whether the vector database can retrieve relevant information for a given query.

In [ ]:
query = "What is Machine Learning?"

results = vectorstore.similarity_search(query)

print(results[0].page_content)

Machine Learning is a subset of Artificial Intelligence where systems learn patterns from data and improve their performance over time without being explicitly programmed.
